In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [2]:
# Parameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
NUM_CLASSES = 8  
#DATASET_DIR = "C:/Users/NAYAN ADHIKARI/Skin-disease-detection/data/skin-disease-datasaet/train_set"


In [3]:
train_dir = "C:/Users/NAYAN ADHIKARI/Skin-disease-detection/data/skin-disease-datasaet/train_set"
val_dir = "C:/Users/NAYAN ADHIKARI/Skin-disease-detection/data/skin-disease-datasaet/test_set"

In [4]:
# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2
)

In [5]:
val_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 924 images belonging to 8 classes.


In [7]:
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 233 images belonging to 8 classes.


In [12]:
# Build model with MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

In [13]:
model = Model(inputs=base_model.input, outputs=predictions)

In [14]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

Epoch 1/10


c:\Users\NAYAN ADHIKARI\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.1481 - loss: 2.4655 - val_accuracy: 0.2103 - val_loss: 2.1126
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.1931 - loss: 2.2141 - val_accuracy: 0.2961 - val_loss: 1.9190
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.2676 - loss: 2.0407 - val_accuracy: 0.3562 - val_loss: 1.7476
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.3536 - loss: 1.7643 - val_accuracy: 0.4163 - val_loss: 1.5950
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.4058 - loss: 1.6776 - val_accuracy: 0.4850 - val_loss: 1.4633
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.4269 - loss: 1.5825 - val_accuracy: 0.5451 - val_loss: 1.3536
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.4793 - loss: 1.4732 - val_accuracy: 0.5751 - val_loss: 1.2562
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.5322 - loss: 1.3424 - val_accuracy: 0.6180 - val_loss: 1.1732
Epo

In [ ]:
base_model.trainable = True

In [ ]:
fine_tune_at = 100

In [ ]:
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

In [ ]:

# Re-compile with a lower learning rate
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:

# Continue training
fine_tune_epochs = 10
total_epochs = EPOCHS + fine_tune_epochs

In [ ]:
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1]
)

Epoch 10/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.3262 - loss: 1.8825 - val_accuracy: 0.7082 - val_loss: 0.9396
Epoch 11/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.4676 - loss: 1.5832 - val_accuracy: 0.7425 - val_loss: 0.8586
Epoch 12/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.5939 - loss: 1.2533 - val_accuracy: 0.7468 - val_loss: 0.8020
Epoch 13/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.6124 - loss: 1.1621 - val_accuracy: 0.7639 - val_loss: 0.7515
Epoch 14/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.7241 - loss: 0.9784 - val_accuracy: 0.7768 - val_loss: 0.7114
Epoch 15/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.7561 - loss: 0.8281 - val_accuracy: 0.7768 - val_loss: 0.6813
Epoch 16/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.7817 - loss: 0.7570 - val_accuracy: 0.7768 - val_loss: 0.6569
Epoch 17/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.8243 - loss: 0.6627 - val_accuracy: 0.7854 - v

In [ ]:
# SAVE FINAL MODEL
model.save("skin_disease_classifier_final.h5")

In [ ]:
# EVALUATE
loss, acc = model.evaluate(val_generator)
print(f"\nFinal Validation Accuracy: {acc:.4f}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.7832 - loss: 0.6346

Final Validation Accuracy: 0.7983
